# SNJ — Original Hybrid RAG coursework

**Trương Minh Hoàng · MSSV 25410008**

**ARCHIVED_SOURCE — source review, not a fresh benchmark.** Bản học tập lịch sử được giữ để giải thích quá trình làm bài. Output cũ và thông tin môi trường riêng đã được bỏ; xem README/AUDIT của dự án trước khi chạy. Historical calculations and known limitations are documented separately.

# SNJ Hybrid RAG Jewelry Advisor — Colab Demo

Notebook đồ án **RAG-based Jewelry Advisor for SNJ**.
Phiên bản này tập trung vào **retrieval quality** thay vì chỉ dùng LLM:

- Intent routing theo nhóm sản phẩm.
- Attribute extraction: đá, màu, size, giới tính, ngân sách, kiểm định.
- Metadata filtering trước khi search.
- Dense semantic search bằng SentenceTransformer.
- Keyword search bằng TF-IDF.
- Hybrid scoring + reranking theo rule domain.
- Template answer không cần LLM.
- Gemini API là tùy chọn, chỉ dùng context đã retrieve.
- Evaluation: category accuracy, Top-1, Top-3, MRR.

Mục tiêu: hạn chế ảo giác, trả đúng nhóm sản phẩm, giải thích được vì sao sản phẩm được đề xuất.

## 1. Cài thư viện

In [ ]:
# Install requirements-dense.txt before running.

## 2. Upload và giải nén project zip

Chọn file `SNJ_RAG_Jewelry_Advisor_Project_HYBRID_RAG.zip` hoặc zip project tương đương.

In [ ]:
import os
from pathlib import Path
# Set SNJ_PROJECT_DIR to your local original project folder.
project_dir = Path(os.environ.get("SNJ_PROJECT_DIR", "data/local"))
print("Project directory:", project_dir)

## 3. Load dataset sản phẩm

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

csv_path = project_dir / 'data/processed/products_enriched.csv'
df = pd.read_csv(csv_path)
print('Dataset shape:', df.shape)
display(df.head(5))

## 4. Kiểm tra schema tối thiểu

In [ ]:
required_cols = ['product_id', 'name', 'category', 'price', 'description', 'search_text']
missing = [c for c in required_cols if c not in df.columns]
print('Missing columns:', missing)
print(df[required_cols].head(3).to_string(index=False))

## 5. Build dense embedding và keyword index

- Dense embedding: hiểu ngữ nghĩa gần đúng.
- TF-IDF keyword: bắt keyword chính xác như `ruby`, `8ly`, `PNJ`, `cặp`, `mặt dây`.

Hybrid RAG cần cả hai, vì dữ liệu sản phẩm trang sức phụ thuộc rất mạnh vào keyword domain.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
model = SentenceTransformer(model_name)

texts = df['search_text'].fillna('').astype(str).tolist()
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True, batch_size=64)
embeddings = np.asarray(embeddings)
print('Dense embeddings shape:', embeddings.shape)

# TF-IDF keyword index. Dùng ngram để bắt cụm như 'thạch anh', 'mắt hổ', 'mặt dây'.
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 3),
    min_df=1,
    max_df=0.95
)
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
tfidf_matrix = normalize(tfidf_matrix)
print('TF-IDF matrix shape:', tfidf_matrix.shape)

## 6. Domain parsing: intent + attribute extraction

Phần này là điểm nâng cấp chính: hệ thống hiểu loại sản phẩm trước, sau đó mới search.

Nguyên tắc:

- Lắc đá/lắc tay/chuỗi đá → `bracelet`.
- Đá rời/làm nhẫn/làm mặt dây/làm bông tai/carat → `loose_stone`.
- Vòng đá/vòng cẩm thạch/vòng nguyên khối → `jade_bangle`.
- Nhẫn/mặt dây/bông tai thành phẩm → đúng nhóm thành phẩm.


In [ ]:
import re
import unicodedata

def normalize_vi(text):
    text = str(text).lower().strip()
    text = unicodedata.normalize('NFD', text)
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    text = text.replace('đ', 'd')
    text = re.sub(r'\s+', ' ', text)
    return text

def phrase_in_text(text, phrase):
    t = normalize_vi(text)
    p = normalize_vi(phrase)
    if not p:
        return False
    # Unicode \w với tiếng Việt đã bỏ dấu, dùng ranh giới để tránh 'cap' khớp nhầm trong 'cao cap'.
    pattern = r'(?<!\w)' + re.escape(p) + r'(?!\w)'
    return re.search(pattern, t) is not None

def contains_any_phrase(text, keywords):
    return any(phrase_in_text(text, k) for k in keywords)

PRODUCT_INTENT_RULES = {
    'stone_bracelet': {
        'categories': ['bracelet'],
        'keywords': ['lắc đá', 'lắc tay', 'chuỗi đá', 'chuỗi đeo tay', 'vòng tay đá', 'bracelet'],
        'boost_terms': ['lắc đá', 'lắc tay', 'chuỗi đá', 'lắc', 'chuỗi']
    },
    'jade_bangle': {
        'categories': ['jade_bangle'],
        'keywords': ['vòng đá', 'vòng cẩm thạch', 'vòng nguyên khối', 'vòng jade', 'vòng ngọc'],
        'boost_terms': ['vòng đá', 'vòng cẩm thạch', 'vòng']
    },
    'loose_stone': {
        'categories': ['loose_stone'],
        'keywords': [
            'đá rời', 'viên đá', 'cặp đá', 'đôi đá',
            'đá làm nhẫn', 'làm nhẫn', 'gắn nhẫn',
            'đá làm mặt dây', 'làm mặt dây', 'gắn mặt dây',
            'đá làm bông tai', 'làm bông tai', 'gắn bông tai',
            'carat', 'ct'
        ],
        'boost_terms': ['viên', 'cặp đá', 'đôi đá', 'ct', 'carat']
    },
    'ring': {
        'categories': ['ring'],
        'keywords': ['nhẫn', 'nhẫn nam', 'nhẫn nữ'],
        'boost_terms': ['nhẫn']
    },
    'pendant': {
        'categories': ['pendant'],
        'keywords': ['mặt dây', 'mặt dây chuyền', 'mặt phật', 'mặt tỳ hưu', 'dây chuyền'],
        'boost_terms': ['mặt dây', 'mặt dây chuyền', 'mặt phật', 'dây chuyền']
    },
    'earrings': {
        'categories': ['earrings'],
        'keywords': ['bông tai', 'hoa tai', 'khuyên tai'],
        'boost_terms': ['bông tai', 'hoa tai', 'khuyên tai']
    }
}

STONE_SYNONYMS = {
    'ruby': ['ruby', 'hồng ngọc', 'hong ngoc'],
    'sapphire': ['sapphire'],
    'citrine': ['citrine', 'thạch anh vàng', 'thach anh vang'],
    'amethyst': ['amethyst', 'thạch anh tím', 'thach anh tim'],
    'aquamarine': ['aquamarine'],
    'garnet': ['garnet'],
    'moonstone': ['moonstone', 'moonstones', 'đá mặt trăng', 'da mat trang'],
    'chalcedony': ['chalcedony'],
    'tiger_eye': ['mắt hổ', 'mat ho', "tiger's eye", 'tiger eye'],
    'jade': ['cẩm thạch', 'cam thach', 'jade'],
    'quartz_hair': ['thạch anh tóc', 'thach anh toc'],
    'topaz': ['topaz'],
    'opal': ['opal'],
    'onyx': ['onyx'],
}

COLOR_TERMS = {
    'vàng': ['vàng', 'vang'],
    'đen': ['đen', 'den'],
    'đỏ': ['đỏ', 'do'],
    'tím': ['tím', 'tim'],
    'xanh': ['xanh'],
    'trắng': ['trắng', 'trang'],
    'nâu': ['nâu', 'nau'],
}

GENDER_TERMS = {
    'male': ['nam', 'cho nam', 'đàn ông'],
    'female': ['nữ', 'cho nữ', 'phụ nữ', 'chị em']
}

def detect_product_intent(query):
    loose_priority = [
        'đá rời', 'viên đá', 'cặp đá', 'đôi đá',
        'đá làm nhẫn', 'làm nhẫn', 'gắn nhẫn',
        'đá làm mặt dây', 'làm mặt dây', 'gắn mặt dây',
        'đá làm bông tai', 'làm bông tai', 'gắn bông tai',
        'carat', 'ct'
    ]
    if contains_any_phrase(query, loose_priority):
        return 'loose_stone'
    if contains_any_phrase(query, PRODUCT_INTENT_RULES['jade_bangle']['keywords']):
        return 'jade_bangle'
    if contains_any_phrase(query, PRODUCT_INTENT_RULES['stone_bracelet']['keywords']):
        return 'stone_bracelet'
    for intent in ['ring', 'pendant', 'earrings']:
        if contains_any_phrase(query, PRODUCT_INTENT_RULES[intent]['keywords']):
            return intent
    return None

def extract_query_sizes(query):
    q = normalize_vi(query)
    sizes = []
    # 9ly5 => 9.5
    for m in re.finditer(r'(\d{1,2})(?:\s*)ly(?:\s*)?(\d)?', q):
        val = float(m.group(1)) + (float('0.' + m.group(2)) if m.group(2) else 0.0)
        sizes.append(val)
    # size 57 / tay 52, dùng cho vòng cẩm thạch
    for m in re.finditer(r'(?:size|tay)\s*(\d{2}(?:[,.]\d+)?)\b', q):
        sizes.append(float(m.group(1).replace(',', '.')))
    return sorted(set(sizes))

def extract_budget(query):
    q = normalize_vi(query)
    # dưới 2 triệu / duoi 2 trieu
    m = re.search(r'(?:duoi|<|nho hon|khong qua)\s*(\d+(?:[,.]\d+)?)\s*(?:trieu|tr)', q)
    if m:
        return int(float(m.group(1).replace(',', '.')) * 1_000_000)
    m = re.search(r'(?:duoi|<|nho hon|khong qua)\s*(\d{5,})', q)
    if m:
        return int(m.group(1))
    return None

def extract_query_domain_terms(query):
    stones, colors, genders = [], [], []
    for canonical, aliases in STONE_SYNONYMS.items():
        if contains_any_phrase(query, aliases):
            stones.append(canonical)
    for color, aliases in COLOR_TERMS.items():
        if contains_any_phrase(query, aliases):
            colors.append(color)
    for gender, aliases in GENDER_TERMS.items():
        if contains_any_phrase(query, aliases):
            genders.append(gender)
    return stones, colors, genders

def parse_query(query):
    return {
        'product_intent': detect_product_intent(query),
        'stones': extract_query_domain_terms(query)[0],
        'colors': extract_query_domain_terms(query)[1],
        'genders': extract_query_domain_terms(query)[2],
        'sizes': extract_query_sizes(query),
        'max_price': extract_budget(query),
        'certificate_required': contains_any_phrase(query, ['kiểm định', 'chứng nhận', 'PNJ', 'SJC']),
        'earring_purpose': contains_any_phrase(query, ['bông tai', 'hoa tai', 'khuyên tai', 'làm bông tai', 'gắn bông tai']),
        'ring_or_pendant_purpose': contains_any_phrase(query, ['làm nhẫn', 'gắn nhẫn', 'làm mặt dây', 'gắn mặt dây']),
    }

for q in ['tìm lắc đá nữ size 8ly màu vàng dưới 1 triệu', 'cặp đá ruby làm bông tai', 'vòng cẩm thạch kiểm định PNJ size 52']:
    print(q, '=>', parse_query(q))

## 7. Metadata filter + hybrid scoring + reranking

Công thức điểm cuối:

```text
final_score = 0.40*dense + 0.25*keyword + 0.30*rule + 0.05*business
```

Trong đó:
- `dense`: embedding/cosine.
- `keyword`: TF-IDF lexical match.
- `rule`: đúng đá, màu, size, mục đích, kiểm định.
- `business`: còn có thể dùng tồn kho/giá/ưu tiên bán sau này.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def row_text_for_match(row):
    parts = [
        row.get('name', ''), row.get('search_text', ''), row.get('stone_names', ''),
        row.get('color', ''), row.get('description', ''), row.get('category', ''),
        row.get('certificate', ''), row.get('certificate_code', '')
    ]
    return ' '.join(str(x) for x in parts if str(x) != 'nan')

def name_has_size(name, target_size):
    n = normalize_vi(name)
    if abs(target_size - round(target_size)) > 0:
        whole = int(target_size)
        dec = int(round((target_size - whole) * 10))
        if re.search(rf'(?<!\d){whole}\s*ly\s*{dec}\b', n):
            return True
    # lắc 8ly / size 8; vòng cẩm thạch size 52
    if re.search(rf'(?<!\d){int(target_size)}\s*ly\b', n):
        return True
    if re.search(rf'\bsize\s*{target_size:g}\b', n):
        return True
    return False

def apply_metadata_filter(dataframe, query_profile):
    result = dataframe.copy()
    intent = query_profile['product_intent']
    result['_detected_intent'] = intent or 'general'

    # Lọc cứng category nếu đã detect product intent.
    if intent is not None:
        cats = PRODUCT_INTENT_RULES[intent]['categories']
        result = result[result['category'].astype(str).isin(cats)].copy()

    # Lọc ngân sách nếu query có budget.
    if query_profile.get('max_price') is not None and 'price' in result.columns:
        result = result[result['price'].fillna(0) <= query_profile['max_price']].copy()

    # Đá rời phải có đơn vị ct nếu query nói carat/ct hoặc mục đích làm trang sức.
    if intent == 'loose_stone':
        if query_profile.get('earring_purpose') or query_profile.get('ring_or_pendant_purpose') or contains_any_phrase(' '.join(map(str, query_profile.values())), ['ct', 'carat']):
            if 'weight_unit' in result.columns:
                result = result[result['weight_unit'].fillna('').astype(str).str.lower().eq('ct')].copy()

    return result

def rule_score(row, query, query_profile):
    text = row_text_for_match(row)
    name = row.get('name', '')
    score = 0.0
    max_score = 1.0

    intent = query_profile.get('product_intent')
    if intent:
        # category đúng đã filter, vẫn cho điểm nền.
        score += 0.20
        for kw in PRODUCT_INTENT_RULES[intent].get('boost_terms', []):
            if phrase_in_text(name, kw):
                score += 0.08
                break

    # Đá/màu là các thuộc tính rất quan trọng.
    for stone in query_profile.get('stones', []):
        aliases = STONE_SYNONYMS[stone]
        if contains_any_phrase(text, aliases):
            score += 0.22
    for color in query_profile.get('colors', []):
        aliases = COLOR_TERMS[color]
        if contains_any_phrase(text, aliases):
            score += 0.18

    # Size chính xác.
    for sz in query_profile.get('sizes', []):
        if name_has_size(name, sz):
            score += 0.25
            break

    # Cặp/đôi cho nhu cầu bông tai.
    if query_profile.get('earring_purpose'):
        if contains_any_phrase(name, ['cặp', 'đôi', 'bông tai']):
            score += 0.25
        elif contains_any_phrase(name, ['viên']):
            score -= 0.10

    # Làm nhẫn/mặt dây thì ưu tiên viên có 'làm nhẫn/mặt dây'.
    if query_profile.get('ring_or_pendant_purpose'):
        if contains_any_phrase(name, ['làm nhẫn', 'mặt dây']):
            score += 0.18

    # Kiểm định.
    if query_profile.get('certificate_required'):
        cert = str(row.get('certificate', '')) + ' ' + str(row.get('certificate_code', '')) + ' ' + name
        if contains_any_phrase(cert, ['PNJ', 'SJC', 'kiểm định', 'chứng nhận']):
            score += 0.18

    return float(max(0.0, min(max_score, score)))

def minmax_norm(x):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return x
    mn, mx = np.nanmin(x), np.nanmax(x)
    if np.isclose(mx - mn, 0):
        return np.zeros_like(x)
    return (x - mn) / (mx - mn)

def retrieve_products_hybrid(query, top_k=5, candidate_k=50, debug=False):
    qp = parse_query(query)
    candidates = apply_metadata_filter(df, qp)
    if len(candidates) == 0:
        cols = ['product_id', 'name', 'category', 'price', 'description']
        empty = df.iloc[0:0][cols].copy()
        empty['detected_intent'] = qp['product_intent'] or 'general'
        return empty

    idx = candidates.index.to_numpy()

    # Dense score
    q_dense = model.encode([query], normalize_embeddings=True)[0]
    dense_scores = embeddings[idx] @ q_dense

    # Keyword TF-IDF score
    q_tfidf = tfidf_vectorizer.transform([query])
    q_tfidf = normalize(q_tfidf)
    keyword_scores = (tfidf_matrix[idx] @ q_tfidf.T).toarray().ravel()

    candidates = candidates.copy()
    candidates['dense_score'] = dense_scores
    candidates['keyword_score_raw'] = keyword_scores
    candidates['rule_score'] = candidates.apply(lambda r: rule_score(r, query, qp), axis=1)
    candidates['business_score'] = 0.0

    # Normalize dense/keyword trong candidate pool để scale ổn định.
    candidates['dense_score_n'] = minmax_norm(candidates['dense_score'].values)
    candidates['keyword_score_n'] = minmax_norm(candidates['keyword_score_raw'].values)

    # Hybrid score.
    candidates['final_score'] = (
        0.40 * candidates['dense_score_n'] +
        0.25 * candidates['keyword_score_n'] +
        0.30 * candidates['rule_score'] +
        0.05 * candidates['business_score']
    )
    candidates['detected_intent'] = qp['product_intent'] or 'general'
    candidates['parsed_query'] = str(qp)

    cols = [
        'product_id', 'name', 'category', 'price', 'weight_value', 'weight_unit',
        'description', 'dense_score_n', 'keyword_score_n', 'rule_score', 'final_score',
        'detected_intent', 'parsed_query'
    ]
    cols = [c for c in cols if c in candidates.columns]
    result = candidates.sort_values('final_score', ascending=False)[cols].head(top_k)

    if debug:
        print('Parsed query:', qp)
        print('Candidate count after metadata filter:', len(candidates))
        if qp['product_intent']:
            print('Allowed categories:', PRODUCT_INTENT_RULES[qp['product_intent']]['categories'])
    return result

## 8. Test truy vấn nâng cao

In [ ]:
test_queries = [
    'tìm lắc đá nữ size 8ly màu vàng',
    'lắc tay đá nam màu đen hoặc mắt hổ',
    'chuỗi đá thạch anh tím đeo tay',
    'tôi muốn đá ruby làm nhẫn hoặc mặt dây',
    'đá rời làm bông tai có carat',
    'cặp đá ruby làm bông tai',
    'vòng đá cẩm thạch size tay nữ',
    'vòng cẩm thạch kiểm định PNJ',
    'nhẫn bạc nam đá garnet',
    'mặt dây đá thạch anh tóc vàng',
    'bông tai bạc đá sapphire',
    'lắc đá nữ size 8ly màu vàng dưới 1 triệu',
]

for q in test_queries:
    print('='*120)
    print('QUERY:', q)
    rows = retrieve_products_hybrid(q, top_k=5, debug=True)
    show_cols = ['product_id', 'name', 'category', 'price', 'weight_value', 'weight_unit', 'dense_score_n', 'keyword_score_n', 'rule_score', 'final_score', 'detected_intent']
    show_cols = [c for c in show_cols if c in rows.columns]
    display(rows[show_cols])

## 9. Evaluation có expected labels

Bảng đánh giá retrieval.

In [ ]:
eval_cases = [
    {
        'query': 'tìm lắc đá nữ size 8ly màu vàng',
        'expected_category': 'bracelet',
        'must_include_any': ['citrine', 'thạch anh vàng', 'vàng', '8ly']
    },
    {
        'query': 'lắc tay đá nam màu đen hoặc mắt hổ',
        'expected_category': 'bracelet',
        'must_include_any': ['mắt hổ', 'xanh đen', 'đen']
    },
    {
        'query': 'chuỗi đá thạch anh tím đeo tay',
        'expected_category': 'bracelet',
        'must_include_any': ['amethyst', 'thạch anh tím']
    },
    {
        'query': 'tôi muốn đá ruby làm nhẫn hoặc mặt dây',
        'expected_category': 'loose_stone',
        'must_include_any': ['ruby', 'làm nhẫn', 'mặt dây']
    },
    {
        'query': 'cặp đá ruby làm bông tai',
        'expected_category': 'loose_stone',
        'must_include_any': ['cặp', 'ruby', 'bông tai']
    },
    {
        'query': 'vòng cẩm thạch kiểm định PNJ',
        'expected_category': 'jade_bangle',
        'must_include_any': ['cẩm thạch', 'pnj']
    },
    {
        'query': 'nhẫn bạc nam đá garnet',
        'expected_category': 'ring',
        'must_include_any': ['nhẫn', 'garnet', 'nam']
    },
    {
        'query': 'mặt dây đá thạch anh tóc vàng',
        'expected_category': 'pendant',
        'must_include_any': ['mặt dây', 'thạch anh tóc vàng']
    },
    {
        'query': 'bông tai bạc đá sapphire',
        'expected_category': 'earrings',
        'must_include_any': ['bông tai', 'sapphire']
    },
]

def row_matches_keywords(row, keywords):
    text = row_text_for_match(row)
    return any(contains_any_phrase(text, [kw]) for kw in keywords)

def evaluate_retrieval(eval_cases, top_k=5):
    records = []
    for case in eval_cases:
        q = case['query']
        rows = retrieve_products_hybrid(q, top_k=top_k)
        expected_cat = case['expected_category']
        keywords = case.get('must_include_any', [])
        category_valid = len(rows) > 0 and rows['category'].eq(expected_cat).all()
        top1_category = len(rows) > 0 and rows.iloc[0]['category'] == expected_cat
        top1_keyword = len(rows) > 0 and row_matches_keywords(rows.iloc[0], keywords)
        top3_keyword = any(row_matches_keywords(r, keywords) for _, r in rows.head(3).iterrows()) if len(rows) else False
        # MRR theo keyword match trong đúng category
        rr = 0.0
        for rank, (_, r) in enumerate(rows.iterrows(), start=1):
            if r['category'] == expected_cat and row_matches_keywords(r, keywords):
                rr = 1.0 / rank
                break
        records.append({
            'query': q,
            'expected_category': expected_cat,
            'detected_intent': rows['detected_intent'].iloc[0] if len(rows) else None,
            'category_valid@k': category_valid,
            'top1_category': top1_category,
            'top1_keyword': top1_keyword,
            'top3_keyword': top3_keyword,
            'mrr_keyword': rr,
            'top1_name': rows.iloc[0]['name'] if len(rows) else None
        })
    ev = pd.DataFrame(records)
    summary = {
        'category_accuracy@k': ev['category_valid@k'].mean(),
        'top1_category_accuracy': ev['top1_category'].mean(),
        'top1_keyword_accuracy': ev['top1_keyword'].mean(),
        'top3_keyword_accuracy': ev['top3_keyword'].mean(),
        'mrr_keyword_mean': ev['mrr_keyword'].mean(),
    }
    return ev, summary

eval_df, eval_summary = evaluate_retrieval(eval_cases, top_k=5)
display(eval_df)
print('Evaluation summary:')
for k, v in eval_summary.items():
    print(f'{k}: {v:.3f}')

## 10. Template answer không dùng LLM

Dùng kết quả retrieve đã được lọc và rerank. Cách này hạn chế ảo giác vì câu trả lời chỉ dựa trên dữ liệu sản phẩm.

In [ ]:
def format_price(value):
    try:
        if pd.notna(value) and float(value) > 0:
            return f'{int(float(value)):,} VND'.replace(',', '.')
    except Exception:
        pass
    return 'liên hệ'

def template_answer(query, rows):
    if len(rows) == 0:
        return 'Chưa tìm thấy sản phẩm phù hợp trong dữ liệu hiện có. Bạn có thể mô tả rõ hơn về loại đá, màu, ngân sách hoặc mục đích sử dụng.'
    lines = [f"Dựa trên câu hỏi: '{query}', hệ thống tìm được các sản phẩm phù hợp nhất:"]
    for i, (_, r) in enumerate(rows.iterrows(), 1):
        price = format_price(r.get('price'))
        reason_parts = []
        reason_parts.append(f"đúng nhóm {r.get('category')}")
        if pd.notna(r.get('weight_value')) and pd.notna(r.get('weight_unit')):
            reason_parts.append(f"trọng lượng {r.get('weight_value')} {r.get('weight_unit')}")
        reason_parts.append(f"điểm phù hợp {r.get('final_score', 0):.3f}")
        lines.append(f"{i}. {r['name']} — {price}. Lý do: " + ', '.join(reason_parts) + '.')
    lines.append('')
    lines.append('Lưu ý: Giá, tồn kho và chính sách mua hàng phải được xác minh tại nguồn trước khi sử dụng.')
    return '\n'.join(lines)

query = 'Tôi muốn mua cặp đá ruby làm bông tai, có carat rõ ràng'
rows = retrieve_products_hybrid(query, top_k=5)
print(template_answer(query, rows))

## 11. Tùy chọn: Gemini API chỉ để diễn đạt tự nhiên hơn

Gemini không quyết định sản phẩm. Sản phẩm đã được quyết định bởi retrieval + rule + reranking.

Nếu không có API key, notebook tự fallback về template answer.

In [ ]:
import os

# Cách dùng đúng trên Colab:
# os.environ['GEMINI_API_KEY'] = 'YOUR_API_KEY_HERE'

def gemini_answer(query, rows, model_name=None):
    api_key = os.getenv('GEMINI_API_KEY')
    model_name = model_name or os.getenv('GEMINI_MODEL')
    if not api_key or not model_name:
        return template_answer(query, rows)
    try:
        import google.generativeai as genai
        genai.configure(api_key=api_key)
        context_blocks = []
        for _, r in rows.iterrows():
            context_blocks.append(
                f"Mã: {r.get('product_id')}\n"
                f"Tên: {r.get('name')}\n"
                f"Danh mục: {r.get('category')}\n"
                f"Giá: {format_price(r.get('price'))}\n"
                f"Mô tả: {r.get('description')}\n"
                f"Điểm phù hợp: {r.get('final_score', 0):.3f}"
            )
        context = '\n\n---\n\n'.join(context_blocks)
        prompt = f'''Bạn là trợ lý tư vấn sản phẩm trang sức đá quý SNJ.
Chỉ dùng dữ liệu trong CONTEXT. Nếu thiếu thông tin thì nói chưa đủ dữ liệu, tuyệt đối không bịa.

QUESTION: {query}

CONTEXT:
{context}

Yêu cầu:
- Trả lời tiếng Việt ngắn gọn.
- Liệt kê 3-5 sản phẩm phù hợp gồm tên, giá, mã nếu có.
- Giải thích lý do dựa trên dữ liệu retrieve.
- Không thêm công dụng phong thủy nếu context không có.
'''
        response = genai.GenerativeModel(model_name).generate_content(prompt)
        return response.text
    except Exception as e:
        print('Gemini error, fallback template:', e)
        return template_answer(query, rows)

query = 'Tôi muốn lắc tay đá nam màu đen hoặc mắt hổ'
rows = retrieve_products_hybrid(query, top_k=5)
print(gemini_answer(query, rows))

## 12. Xuất kết quả demo và evaluation ra CSV

In [ ]:
demo_queries = [
    'tìm lắc đá nữ size 8ly màu vàng',
    'lắc tay đá nam màu đen hoặc mắt hổ',
    'chuỗi đá thạch anh tím đeo tay',
    'đá ruby làm nhẫn có kiểm định',
    'cặp đá ruby làm bông tai',
    'vòng đá cẩm thạch kiểm định PNJ',
    'nhẫn bạc nam đá garnet',
    'mặt dây đá thạch anh tóc vàng',
    'bông tai bạc đá sapphire'
]

all_results = []
for q in demo_queries:
    rows = retrieve_products_hybrid(q, top_k=5)
    tmp = rows.copy()
    tmp.insert(0, 'query', q)
    all_results.append(tmp)

out = pd.concat(all_results, ignore_index=True)
Path('artifacts').mkdir(exist_ok=True)
out_path = 'artifacts/snj_hybrid_rag_demo_results.csv'
eval_path = 'artifacts/snj_hybrid_rag_evaluation.csv'
out.to_csv(out_path, index=False, encoding='utf-8-sig')
eval_df.to_csv(eval_path, index=False, encoding='utf-8-sig')

print('Saved:', out_path)
print('Saved:', eval_path)

# Output files remain local; no upload/download side effects.

## 13. Kết luận cho báo cáo

Phiên bản Hybrid RAG này tốt hơn bản cosine-only vì:

1. Query được hiểu intent trước khi search.
2. Metadata filter hạn chế trả sai nhóm sản phẩm.
3. Dense embedding xử lý ngữ nghĩa gần đúng.
4. TF-IDF keyword giữ độ chính xác với tên đá, size, mã kiểm định, đơn vị carat.
5. Reranking bằng rule domain giúp đưa sản phẩm đúng mục đích lên đầu.
6. Có evaluation định lượng: category accuracy, top-k keyword accuracy, MRR.
7. LLM là tùy chọn và chỉ dùng để diễn đạt lại context, không tự bịa sản phẩm.